# Physics-Conditioned ANC Net (Cross-Path Version)

本 Notebook 面向"自通路 + 交叉次级通路"联合建模，核心思路：
1. `x_gcc` 作为共享参考分支输入（共享 xref 信息）
2. `S_paths_full` 拆分为 `self path` 与 `cross path` 两个条件分支
3. 预测两个不同输出滤波器系数：`c_self` 与 `c_cross`
4. 使用物理感知损失（系数空间 + 重构空间）联合训练

In [ ]:
import random
from pathlib import Path

import h5py
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import clear_output

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, random_split

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device =', device)

In [ ]:
# =========================
# 数据与特征工具函数
# =========================
def resize_gcc_batch(gcc: np.ndarray, target_len: int = 256) -> np.ndarray:
    """将 [N, 3, L] 插值到 [N, 3, target_len]。"""
    n, c, l = gcc.shape
    if l == target_len:
        return gcc.astype(np.float32)

    old_grid = np.linspace(0.0, 1.0, l, dtype=np.float32)
    new_grid = np.linspace(0.0, 1.0, target_len, dtype=np.float32)
    out = np.zeros((n, c, target_len), dtype=np.float32)
    for i in range(n):
        for ch in range(c):
            out[i, ch] = np.interp(new_grid, old_grid, gcc[i, ch]).astype(np.float32)
    return out

def split_self_cross_tensor(x_full: np.ndarray) -> tuple[np.ndarray, np.ndarray]:
    """
    输入 x_full: [N, 3, 3, L]
    输出:
      self_vec:  [N, 3*L]
      cross_vec: [N, 6*L]
    """
    n, n_sec, n_ref, l = x_full.shape
    self_list = []
    cross_list = []
    for sec in range(n_sec):
        self_list.append(x_full[:, sec, sec, :])
        for ref in range(n_ref):
            if ref != sec:
                cross_list.append(x_full[:, sec, ref, :])

    self_arr = np.stack(self_list, axis=1)
    cross_arr = np.stack(cross_list, axis=1)

    return self_arr.reshape(n, -1).astype(np.float32), cross_arr.reshape(n, -1).astype(np.float32)

def fit_svd_basis(x: np.ndarray, n_components: int = 32):
    """对样本矩阵 x:[N,D] 拟合SVD基底，返回 coeff/mean/components。"""
    x = np.asarray(x, dtype=np.float32)
    mean = x.mean(axis=0, keepdims=True)
    xc = x - mean
    _, _, vt = np.linalg.svd(xc, full_matrices=False)
    k = min(n_components, vt.shape[0])
    comp = vt[:k].astype(np.float32)
    coeff = (xc @ comp.T).astype(np.float32)
    return coeff, mean.reshape(-1).astype(np.float32), comp

In [ ]:
# =========================
# 读取HDF5并构建训练张量
# =========================
candidate = Path('../python_scripts/cfxlms_qc_dataset_cross_500.h5')
DATASET_H5 = candidate if candidate.exists() else Path('../python_scripts/cfxlms_qc_dataset_cross_diag30.h5')
print('DATASET_H5 =', DATASET_H5.resolve())

with h5py.File(str(DATASET_H5), 'r') as f:
    x_gcc_raw = np.asarray(f['/processed/gcc_phat'], dtype=np.float32)

    if '/raw/S_paths_full' not in f:
        raise KeyError('缺少 /raw/S_paths_full，无法训练跨通路网络。')
    if '/raw/W_full' not in f:
        raise KeyError('缺少 /raw/W_full，无法训练跨通路网络。')

    s_full = np.asarray(f['/raw/S_paths_full'], dtype=np.float32)
    w_full = np.asarray(f['/raw/W_full'], dtype=np.float32)

x_gcc = resize_gcc_batch(x_gcc_raw, target_len=256)
x_s_self, x_s_cross = split_self_cross_tensor(s_full)
w_self_wave, w_cross_wave = split_self_cross_tensor(w_full)

# 使用SVD构建 self/cross 两套输出系数与物理基底
y_self_coef, w_self_mean, V_w_self = fit_svd_basis(w_self_wave, n_components=32)
y_cross_coef, w_cross_mean, V_w_cross = fit_svd_basis(w_cross_wave, n_components=32)

print('x_gcc:', x_gcc.shape)
print('x_s_self:', x_s_self.shape)
print('x_s_cross:', x_s_cross.shape)
print('y_self_coef:', y_self_coef.shape)
print('y_cross_coef:', y_cross_coef.shape)
print('V_w_self:', V_w_self.shape)
print('V_w_cross:', V_w_cross.shape)

In [ ]:
# =========================
# Dataset & DataLoader
# =========================
class CrossPathANCDataset(Dataset):
    def __init__(self, x_gcc, x_s_self, x_s_cross, y_self_coef, y_cross_coef):
        self.x_gcc = torch.from_numpy(np.asarray(x_gcc, dtype=np.float32))
        self.x_s_self = torch.from_numpy(np.asarray(x_s_self, dtype=np.float32))
        self.x_s_cross = torch.from_numpy(np.asarray(x_s_cross, dtype=np.float32))
        self.y_self_coef = torch.from_numpy(np.asarray(y_self_coef, dtype=np.float32))
        self.y_cross_coef = torch.from_numpy(np.asarray(y_cross_coef, dtype=np.float32))

    def __len__(self):
        return self.x_gcc.shape[0]

    def __getitem__(self, idx):
        return (
            self.x_gcc[idx],
            self.x_s_self[idx],
            self.x_s_cross[idx],
            self.y_self_coef[idx],
            self.y_cross_coef[idx],
        )

dataset = CrossPathANCDataset(x_gcc, x_s_self, x_s_cross, y_self_coef, y_cross_coef)

train_size = int(0.8 * len(dataset))
val_size = len(dataset) - train_size
train_set, val_set = random_split(dataset, [train_size, val_size], generator=torch.Generator().manual_seed(42))

train_loader = DataLoader(train_set, batch_size=64, shuffle=True, num_workers=0, pin_memory=torch.cuda.is_available())
val_loader = DataLoader(val_set, batch_size=64, shuffle=False, num_workers=0, pin_memory=torch.cuda.is_available())

V_w_self_t = torch.tensor(V_w_self, dtype=torch.float32, device=device)
V_w_cross_t = torch.tensor(V_w_cross, dtype=torch.float32, device=device)

print('train/val =', len(train_set), len(val_set))
batch = next(iter(train_loader))
print('batch shapes:', [tuple(t.shape) for t in batch])

In [ ]:
# =========================
# 模型：共享xref + self/cross双条件 + 双输出头
# =========================
class CrossPathPhysicalConditionedANCNet(nn.Module):
    def __init__(self, self_dim: int, cross_dim: int, coef_dim: int = 32):
        super().__init__()

        # 共享参考分支（shared xref）
        self.shared_xref_encoder = nn.Sequential(
            nn.Conv1d(3, 16, kernel_size=7, stride=2, padding=3),
            nn.GroupNorm(1, 16),
            nn.GELU(),
            nn.Conv1d(16, 32, kernel_size=5, stride=2, padding=2),
            nn.GroupNorm(1, 32),
            nn.GELU(),
            nn.Conv1d(32, 64, kernel_size=3, stride=2, padding=1),
            nn.GroupNorm(1, 64),
            nn.GELU(),
            nn.AdaptiveAvgPool1d(1),
            nn.Flatten(),
            nn.Linear(64, 128),
            nn.GELU(),
        )

        # self次级通路编码器
        self.self_path_encoder = nn.Sequential(
            nn.Linear(self_dim, 256),
            nn.LayerNorm(256),
            nn.GELU(),
            nn.Linear(256, 128),
            nn.LayerNorm(128),
            nn.GELU(),
        )

        # cross次级通路编码器
        self.cross_path_encoder = nn.Sequential(
            nn.Linear(cross_dim, 256),
            nn.LayerNorm(256),
            nn.GELU(),
            nn.Linear(256, 128),
            nn.LayerNorm(128),
            nn.GELU(),
        )

        # self输出头：输出 self 滤波器低秩系数
        self.self_head = nn.Sequential(
            nn.Linear(256, 128),
            nn.LayerNorm(128),
            nn.GELU(),
            nn.Dropout(0.2),
            nn.Linear(128, 64),
            nn.GELU(),
            nn.Linear(64, coef_dim),
        )

        # cross输出头：输出 cross 滤波器低秩系数
        self.cross_head = nn.Sequential(
            nn.Linear(256, 128),
            nn.LayerNorm(128),
            nn.GELU(),
            nn.Dropout(0.2),
            nn.Linear(128, 64),
            nn.GELU(),
            nn.Linear(64, coef_dim),
        )

    def forward(self, x_gcc, x_s_self, x_s_cross):
        z_shared = self.shared_xref_encoder(x_gcc)
        z_self = self.self_path_encoder(x_s_self)
        z_cross = self.cross_path_encoder(x_s_cross)

        c_self_pred = self.self_head(torch.cat([z_shared, z_self], dim=1))
        c_cross_pred = self.cross_head(torch.cat([z_shared, z_cross], dim=1))
        return c_self_pred, c_cross_pred

model = CrossPathPhysicalConditionedANCNet(
    self_dim=x_s_self.shape[1],
    cross_dim=x_s_cross.shape[1],
    coef_dim=32,
).to(device)

print(model)

In [ ]:
# =========================
# 损失函数：双头Physics-Aware Loss
# =========================
def dual_physics_aware_loss(
    c_self_pred, c_self_true, c_cross_pred, c_cross_true,
    V_self, V_cross, lambda_phy=0.1
):
    mse_coef = F.mse_loss(c_self_pred, c_self_true) + F.mse_loss(c_cross_pred, c_cross_true)

    w_self_pred = c_self_pred @ V_self
    w_self_true = c_self_true @ V_self
    w_cross_pred = c_cross_pred @ V_cross
    w_cross_true = c_cross_true @ V_cross

    mse_wave = F.mse_loss(w_self_pred, w_self_true) + F.mse_loss(w_cross_pred, w_cross_true)
    loss = mse_coef + lambda_phy * mse_wave
    return loss, mse_coef.detach(), mse_wave.detach()

def plot_live(train_losses, val_losses):
    clear_output(wait=True)
    plt.figure(figsize=(8, 5))
    plt.plot(train_losses, label='train', linewidth=2.0)
    plt.plot(val_losses, label='val', linewidth=2.0)
    plt.xlabel('epoch')
    plt.ylabel('loss')
    plt.title('Cross-Path Physics-Conditioned ANC Training')
    plt.grid(True, alpha=0.3)
    plt.legend()
    plt.show()

In [ ]:
# =========================
# 训练循环
# =========================
epochs = 100
lambda_phy = 0.1

model = CrossPathPhysicalConditionedANCNet(
    self_dim=x_s_self.shape[1],
    cross_dim=x_s_cross.shape[1],
    coef_dim=32,
).to(device)

optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', patience=5, factor=0.5)

best_val = float('inf')
best_path = Path('best_model_cross_paths.pth')

train_losses = []
val_losses = []

for epoch in range(1, epochs + 1):
    model.train()
    train_sum = 0.0

    for x_gcc_b, x_s_self_b, x_s_cross_b, y_self_b, y_cross_b in train_loader:
        x_gcc_b = x_gcc_b.to(device, non_blocking=True)
        x_s_self_b = x_s_self_b.to(device, non_blocking=True)
        x_s_cross_b = x_s_cross_b.to(device, non_blocking=True)
        y_self_b = y_self_b.to(device, non_blocking=True)
        y_cross_b = y_cross_b.to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)
        c_self_pred, c_cross_pred = model(x_gcc_b, x_s_self_b, x_s_cross_b)

        loss, _, _ = dual_physics_aware_loss(
            c_self_pred, y_self_b, c_cross_pred, y_cross_b,
            V_w_self_t, V_w_cross_t,
            lambda_phy=lambda_phy,
        )

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0)
        optimizer.step()

        train_sum += loss.item() * x_gcc_b.size(0)

    train_epoch = train_sum / len(train_loader.dataset)

    model.eval()
    val_sum = 0.0
    with torch.no_grad():
        for x_gcc_b, x_s_self_b, x_s_cross_b, y_self_b, y_cross_b in val_loader:
            x_gcc_b = x_gcc_b.to(device, non_blocking=True)
            x_s_self_b = x_s_self_b.to(device, non_blocking=True)
            x_s_cross_b = x_s_cross_b.to(device, non_blocking=True)
            y_self_b = y_self_b.to(device, non_blocking=True)
            y_cross_b = y_cross_b.to(device, non_blocking=True)

            c_self_pred, c_cross_pred = model(x_gcc_b, x_s_self_b, x_s_cross_b)
            vloss, _, _ = dual_physics_aware_loss(
                c_self_pred, y_self_b, c_cross_pred, y_cross_b,
                V_w_self_t, V_w_cross_t,
                lambda_phy=lambda_phy,
            )
            val_sum += vloss.item() * x_gcc_b.size(0)

    val_epoch = val_sum / len(val_loader.dataset)
    scheduler.step(val_epoch)

    if val_epoch < best_val:
        best_val = val_epoch
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'best_val': best_val,
        }, best_path)

    train_losses.append(train_epoch)
    val_losses.append(val_epoch)
    plot_live(train_losses, val_losses)

    lr_now = optimizer.param_groups[0]['lr']
    print(f'Epoch [{epoch:03d}/{epochs}] train={train_epoch:.6f} val={val_epoch:.6f} best={best_val:.6f} lr={lr_now:.2e}')

print('训练完成，最佳模型:', best_path.resolve())

In [ ]:
# =========================
# 推理演示：重构 self/cross 滤波器向量
# =========================
ckpt = torch.load('best_model_cross_paths.pth', map_location=device)
model.load_state_dict(ckpt['model_state_dict'])
model.eval()

x_gcc_1, x_s_self_1, x_s_cross_1, y_self_1, y_cross_1 = val_set[0]
with torch.no_grad():
    c_self_p, c_cross_p = model(
        x_gcc_1.unsqueeze(0).to(device),
        x_s_self_1.unsqueeze(0).to(device),
        x_s_cross_1.unsqueeze(0).to(device),
    )

c_self_p = c_self_p.squeeze(0)
c_cross_p = c_cross_p.squeeze(0)

w_self_pred = c_self_p @ V_w_self_t
w_cross_pred = c_cross_p @ V_w_cross_t
w_self_true = y_self_1.to(device) @ V_w_self_t
w_cross_true = y_cross_1.to(device) @ V_w_cross_t

print('best epoch =', ckpt['epoch'])
print('best val =', ckpt['best_val'])
print('self recon L2 error =', torch.norm(w_self_pred - w_self_true).item())
print('cross recon L2 error =', torch.norm(w_cross_pred - w_cross_true).item())